# 🚀 Ajuste fino de RoBERTa con GPU
*Curso de NLP · 🚀 Llévalo a Colab · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

**Objetivo:** ajustar **RoBERTa-base** (125 millones de parámetros, MIT) para decidir si una crítica de cine es
positiva o negativa, con el `Trainer` de Hugging Face.

En el curso de NLP ajustaste DistilBETO en CPU con pocos ejemplos. Con la GPU de Colab entrenas un modelo más grande
con miles de críticas en minutos.

**Los datos:** *IMDB* (`stanfordnlp/imdb`): 50.000 críticas de cine en inglés, mitad positivas y mitad negativas.

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"   # solo para la prueba automática de MiAtlas.cl

import torch
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Colab:", EN_COLAB, "| dispositivo:", DISPOSITIVO)
if DISPOSITIVO == "cpu":
    print("⚠ Sin GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.")

In [ ]:
from pathlib import Path
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = Path("/content/drive/MyDrive/MiAtlas-NLP")
else:
    CARPETA = Path("MiAtlas-NLP")
CARPETA.mkdir(parents=True, exist_ok=True)
print("Carpeta de trabajo:", CARPETA)

In [ ]:
%pip install -q transformers datasets evaluate accelerate

In [ ]:
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")
N_TRAIN, N_TEST = (300, 100) if PRUEBA_RAPIDA else (8000, 2000)   # una muestra: la base completa tarda más de 1 h
train = imdb["train"].shuffle(seed=0).select(range(N_TRAIN))
test = imdb["test"].shuffle(seed=0).select(range(N_TEST))
print(train)
print(train[0]["text"][:300], "→", "positiva" if train[0]["label"] else "negativa")

## 1. Tokenizar

El tokenizador de RoBERTa parte el texto en piezas (*subwords*). `truncation` corta las críticas largas a 256 piezas
y `DataCollatorWithPadding` rellena cada lote solo hasta la más larga del lote: más rápido que rellenar todo a 256.

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

BASE = "FacebookAI/roberta-base"
tokenizador = AutoTokenizer.from_pretrained(BASE)
tok_train = train.map(lambda l: tokenizador(l["text"], truncation=True, max_length=256), batched=True)
tok_test = test.map(lambda l: tokenizador(l["text"], truncation=True, max_length=256), batched=True)
rellenar = DataCollatorWithPadding(tokenizador)
print(tokenizador.tokenize("This movie was surprisingly good!"))

## 2. Entrenar con `Trainer`

In [ ]:
import numpy as np
import evaluate
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, set_seed

set_seed(0)
modelo = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=2, id2label={0: "negativa", 1: "positiva"}, label2id={"negativa": 0, "positiva": 1})

f1 = evaluate.load("f1")
exactitud = evaluate.load("accuracy")

def metricas(prediccion):
    logits, etiquetas = prediccion
    pred = np.argmax(logits, axis=1)
    return {**exactitud.compute(predictions=pred, references=etiquetas), **f1.compute(predictions=pred, references=etiquetas)}

argumentos = TrainingArguments(
    output_dir="roberta-imdb", report_to="none",
    eval_strategy="epoch", save_strategy="no",
    learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=32,
    num_train_epochs=2, weight_decay=0.01, warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),          # media precisión en GPU: más rápido y con menos memoria
)
entrenador = Trainer(model=modelo, args=argumentos, train_dataset=tok_train, eval_dataset=tok_test,
                     data_collator=rellenar, compute_metrics=metricas)

In [ ]:
import time
inicio = time.perf_counter()
entrenador.train()
print(f"Entrenamiento: {(time.perf_counter() - inicio) / 60:.1f} min en {DISPOSITIVO}")
entrenador.evaluate()

## 3. Guardar en Drive y usarlo

In [ ]:
destino = CARPETA / "roberta-imdb"
entrenador.save_model(destino)
tokenizador.save_pretrained(destino)

from transformers import pipeline
clasificar = pipeline("text-classification", model=str(destino), device=0 if DISPOSITIVO == "cuda" else -1)
clasificar(["An absolute masterpiece, I cried at the end.", "Two hours of my life I will never get back."])

## Pruébalo tú

1. Cambia `BASE` a `dccuchile/bert-base-spanish-wwm-cased` (**BETO**) y los datos a los tweets en español del tema 8
   del curso de Colab (`mteb/tweet_sentiment_multilingual`, `"spanish"`, 3 clases). ¿Qué cambia en el código?
2. Escribe en `diferencia` los puntos de exactitud que gana RoBERTa sobre una regresión logística con TF-IDF en las
   mismas críticas.

In [ ]:
diferencia = None   # ← tu código

In [ ]:
# Comprobación
if diferencia is None:
    print("Aún no lo resuelves.")
else:
    print(f"✅ RoBERTa gana {diferencia:+.1f} puntos sobre TF-IDF")

## Resumen

- `Trainer` + `TrainingArguments` hacen el ciclo de entrenamiento y la evaluación por época.
- `fp16=True` en GPU acelera y ahorra memoria.
- `save_model` en **Drive** guarda el modelo ajustado; `pipeline` lo usa después.

**Fuentes:** [Transformers · Trainer](https://huggingface.co/docs/transformers/main_classes/trainer) · [Transformers · ajuste fino](https://huggingface.co/docs/transformers/training) · [RoBERTa-base (MIT)](https://huggingface.co/FacebookAI/roberta-base) · [IMDB](https://huggingface.co/datasets/stanfordnlp/imdb) · [Evaluate](https://huggingface.co/docs/evaluate/index)